# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

This notebook turns the CTR Engagement Opportunity model (w05) into a ranked, human-ready action queue. Work the sections **in order**.

> Skills loaded: `building-baselines` · `training-honest-models` · `flyrank/flyrank-data`

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### What this section builds

We re-train the Random Forest from w05 on the same cache and time-split, then score **all 60 k eligible pages** — not just the held-out test clients. Each page receives:

| Column | Meaning |
|---|---|
| `rf_prob` | Model probability that this page will see >= 10 % CTR improvement |
| `priority_score` | `rf_prob x opportunity_gap x log1p(impressions)` — weights confidence by impact and audience size |
| `action_label` | One of four tiers (see table below) |
| `reason_text` | Plain-English explanation a non-technical reviewer can act on |

**Action label thresholds (evaluated in priority order — first match wins):**

| Condition | Label |
|---|---|
| `rf_prob >= 0.70` AND tier in `{top_3, striking}` AND `gap > 0.50 pp` | `rewrite_title_and_meta` |
| `rf_prob >= 0.70` AND tier = `page_1` AND `gap > 0.30 pp` | `review_snippet_and_intent` |
| `rf_prob >= 0.50` (any tier) | `monitor_ctr` |
| Otherwise | `no_action` |

The model's probability is directional decision-support, not a causal guarantee.

In [1]:
# -- 1a. Setup & load cache ---------------------------------------------------
import pathlib, warnings, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore')

try:
    import google.colab  # noqa: F401
    REPO_ROOT = pathlib.Path('/content/flyrank')
except ImportError:
    REPO_ROOT = pathlib.Path.cwd()
    for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
        if (parent / 'work').exists() and (parent / 'skills').exists():
            REPO_ROOT = parent
            break

OUTPUTS   = REPO_ROOT / 'work' / 'outputs'
DEV_CACHE = OUTPUTS / 'hf_features_dev.parquet'

df = pd.read_parquet(DEV_CACHE)
print(f'Cache loaded  : {len(df):,} rows  |  {df["client_hash_id"].nunique()} clients')
print(f'Columns       : {list(df.columns)}')

Cache loaded  : 60,088 rows  |  28 clients
Columns       : ['client_hash_id', 'content_hash_id', 'days_active', 'total_impressions', 'total_clicks', 'feat_ctr', 'avg_position', 'ctr_trend', 'label_ctr', 'label_impressions', 'ctr_improved', 'position_tier', 'tier_expected_ctr', 'opportunity_gap', 'log_impressions', 'baseline_score']


In [2]:
# -- 1b. Reproduce w05 train split and re-train RF ----------------------------
FEATURES = ['log_impressions', 'feat_ctr', 'avg_position',
            'days_active', 'opportunity_gap', 'ctr_trend']
TARGET   = 'ctr_improved'

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, _ = next(gss.split(df, groups=df['client_hash_id']))
train_df = df.iloc[train_idx].copy()

rf = RandomForestClassifier(
    n_estimators=100, max_depth=6, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
rf.fit(train_df[FEATURES].fillna(0), train_df[TARGET])
df['rf_prob'] = rf.predict_proba(df[FEATURES].fillna(0))[:, 1]

print(f'RF trained on : {len(train_df):,} rows  ({train_df["client_hash_id"].nunique()} clients)')
print(f'Scored        : {len(df):,} rows')
print(f'rf_prob range : {df["rf_prob"].min():.3f}  -  {df["rf_prob"].max():.3f}')
print(f'rf_prob >= 0.70 : {(df["rf_prob"] >= 0.70).sum():,} pages')
print(f'rf_prob >= 0.50 : {(df["rf_prob"] >= 0.50).sum():,} pages')

RF trained on : 44,118 rows  (22 clients)
Scored        : 60,088 rows
rf_prob range : 0.041  -  1.000
rf_prob >= 0.70 : 23,916 pages
rf_prob >= 0.50 : 29,291 pages


In [3]:
# -- 1c. Priority score -------------------------------------------------------
df['priority_score'] = (
    df['rf_prob']
    * df['opportunity_gap']
    * np.log1p(df['total_impressions'])
)
print(f'Priority score range  : {df["priority_score"].min():.3f}  -  {df["priority_score"].max():.3f}')
print(f'Priority score median : {df["priority_score"].median():.3f}')

Priority score range  : 0.000  -  25.323
Priority score median : 1.333


In [4]:
# -- 1d. Action labels + reason texts -----------------------------------------
cond_rewrite = (
    (df['rf_prob'] >= 0.70)
    & df['position_tier'].isin(['top_3', 'striking'])
    & (df['opportunity_gap'] > 0.50)
)
cond_review = (
    (df['rf_prob'] >= 0.70)
    & (df['position_tier'] == 'page_1')
    & (df['opportunity_gap'] > 0.30)
)
cond_monitor = (df['rf_prob'] >= 0.50)

df['action_label'] = np.select(
    [cond_rewrite, cond_review, cond_monitor],
    ['rewrite_title_and_meta', 'review_snippet_and_intent', 'monitor_ctr'],
    default='no_action'
)
REASON_MAP = {
    'rewrite_title_and_meta'   : 'High-confidence CTR opportunity: page ranks in top-10 but captures far below tier benchmark -- title/meta rewrite is the highest-leverage lever',
    'review_snippet_and_intent': 'Moderate opportunity on Page 1: CTR gap exists but SERP features may be suppressing clicks -- review featured-snippet eligibility and intent alignment before rewriting',
    'monitor_ctr'              : 'Directional signal present but below confidence threshold -- flag for next review cycle if impression volume grows',
    'no_action'                : 'Performing at or above tier benchmark, or insufficient volume for a reliable signal',
}
df['reason_text'] = df['action_label'].map(REASON_MAP)

dist = df['action_label'].value_counts()
print('-- Action Distribution (all 60 k pages) ---------------------------------')
for label, n in dist.items():
    print(f'  {label:<32s}: {n:>6,}  ({100*n/len(df):.1f}%)')
print()
print('-- Action Distribution by Position Tier ---------------------------------')
print(df.groupby(['position_tier', 'action_label']).size().unstack(fill_value=0).to_string())

-- Action Distribution (all 60 k pages) ---------------------------------
  no_action                       : 30,797  (51.3%)
  monitor_ctr                     : 16,370  (27.2%)
  review_snippet_and_intent       : 10,038  (16.7%)
  rewrite_title_and_meta          :  2,883  (4.8%)

-- Action Distribution by Position Tier ---------------------------------
action_label   monitor_ctr  no_action  review_snippet_and_intent  rewrite_title_and_meta
position_tier                                                                           
deep                  5474       2486                          0                       0
no_data                  2          0                          0                       0
page_1                3535      18057                      10038                       0
page_3_5              2730       2508                          0                       0
striking              3146       3509                          0                       0
top_3                

In [5]:
# -- 1e. Ranked priority queue ------------------------------------------------
queue = df.sort_values('priority_score', ascending=False).reset_index(drop=True)
queue['queue_rank'] = queue.index + 1

DISP = ['queue_rank', 'content_hash_id', 'position_tier', 'avg_position',
        'total_impressions', 'feat_ctr', 'tier_expected_ctr',
        'opportunity_gap', 'rf_prob', 'priority_score', 'action_label']

def fmt(sub):
    s = sub[DISP].copy()
    s['content_hash_id']   = s['content_hash_id'].str[:20]
    s['feat_ctr']          = s['feat_ctr'].map('{:.3f}%'.format)
    s['tier_expected_ctr'] = s['tier_expected_ctr'].map('{:.3f}%'.format)
    s['opportunity_gap']   = s['opportunity_gap'].map('{:.3f} pp'.format)
    s['rf_prob']           = s['rf_prob'].map('{:.3f}'.format)
    s['priority_score']    = s['priority_score'].map('{:.3f}'.format)
    s['total_impressions'] = s['total_impressions'].map('{:,.0f}'.format)
    s['avg_position']      = s['avg_position'].map('{:.1f}'.format)
    return s

print('-- Top 25 Overall Priority Queue ----------------------------------------')
print(fmt(queue).head(25).to_string(index=False))
print()
print('-- Top 25 flagged: rewrite_title_and_meta --------------------------------')
rw = queue[queue['action_label'] == 'rewrite_title_and_meta'].head(25)
print(fmt(rw).to_string(index=False))
print('\nSection 1 complete.')

-- Top 25 Overall Priority Queue ----------------------------------------


 queue_rank      content_hash_id position_tier avg_position total_impressions feat_ctr tier_expected_ctr opportunity_gap rf_prob priority_score           action_label
          1 content_06d45e82b369         top_3          1.7            16,041   0.000%            2.764%        2.764 pp   0.946         25.323 rewrite_title_and_meta
          2 content_39bb094b613d         top_3          1.3             5,552   0.000%            2.764%        2.764 pp   0.973         23.182 rewrite_title_and_meta
          3 content_c0239977f391         top_3          3.0             5,360   0.000%            2.764%        2.764 pp   0.969         23.001 rewrite_title_and_meta
          4 content_b42b2d8b7693         top_3          0.2             5,215   0.000%            2.764%        2.764 pp   0.966         22.855 rewrite_title_and_meta
          5 content_94ea772ab734         top_3          0.4             4,776   0.000%            2.764%        2.764 pp   0.972         22.773 rewrite_title_and_met

 queue_rank      content_hash_id position_tier avg_position total_impressions feat_ctr tier_expected_ctr opportunity_gap rf_prob priority_score           action_label
          1 content_06d45e82b369         top_3          1.7            16,041   0.000%            2.764%        2.764 pp   0.946         25.323 rewrite_title_and_meta
          2 content_39bb094b613d         top_3          1.3             5,552   0.000%            2.764%        2.764 pp   0.973         23.182 rewrite_title_and_meta
          3 content_c0239977f391         top_3          3.0             5,360   0.000%            2.764%        2.764 pp   0.969         23.001 rewrite_title_and_meta
          4 content_b42b2d8b7693         top_3          0.2             5,215   0.000%            2.764%        2.764 pp   0.966         22.855 rewrite_title_and_meta
          5 content_94ea772ab734         top_3          0.4             4,776   0.000%            2.764%        2.764 pp   0.972         22.773 rewrite_title_and_met

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

**Who:** An SEO or editorial team deciding which pages to prioritise for title/meta-description review in the current sprint.

**Decision supported:** *"Given a backlog of underperforming pages, which ones should we send to a writer first?"* — a ranking and triage question, not an automatic rewrite trigger.

**How to use the queue:**
1. Work down the queue from rank #1
2. For each `rewrite_title_and_meta` page, a human reviews intent, current SERP layout, and the existing title before writing a new one
3. For `review_snippet_and_intent` pages, investigate whether a SERP feature (AI Overview, featured snippet) is suppressing clicks before investing in a rewrite
4. `monitor_ctr` pages go on a watchlist for the next refresh cycle — revisit if impressions grow

### Where this stops being valid

| Limitation | Why it matters |
|---|---|
| **Observation window** | Dev split: Jan features -> Feb labels (single-month snapshot). Seasonal patterns, algorithm updates, or SERP layout changes after this window are not captured |
| **Client coverage** | 28 anonymised clients from one warehouse release. Patterns may not transfer to sites in different verticals or at very different scales |
| **Volume floor** | Pages with < 100 impressions per window are excluded — CTR is too noisy below this threshold to produce reliable labels |
| **No causal claim** | The model identifies pages where CTR *was observed to improve*; it cannot guarantee a rewrite *will cause* improvement |
| **SERP features invisible** | AI Overviews, featured snippets, People Also Ask, and rich results suppress organic CTR in ways unavailable in the dataset |
| **Navigational queries** | Branded and navigational pages have structurally different CTR dynamics — this model is not calibrated for them |

In [6]:
# -- 2. Formal intended-use statement (quotable in the paper) -----------------
statement = """
-- Intended Use Statement -------------------------------------------------------
This priority queue is a directional decision-support tool for editorial triage.
It ranks pages by observed CTR opportunity and model confidence, but does not
guarantee that a rewrite will cause improvement. All recommendations require
human review before action.

Validation scope  : 28 clients, Jan-Feb 2026 dev window, >=100 impression floor.
Not valid for     : navigational/branded queries, pages outside the observed
                    volume range, or sites in verticals absent from this release.
Language          : findings are observational and directional; no causal claims.
--------------------------------------------------------------------------------
"""
print(statement)

total      = len(df)
actionable = (df['action_label'] != 'no_action').sum()
rewrite_n  = (df['action_label'] == 'rewrite_title_and_meta').sum()
review_n   = (df['action_label'] == 'review_snippet_and_intent').sum()
monitor_n  = (df['action_label'] == 'monitor_ctr').sum()

print('Scope summary')
print(f'  Total pages scored       : {total:,}')
print(f'  Actionable (any tier)    : {actionable:,}  ({100*actionable/total:.1f}%)')
print(f'  -> rewrite_title_and_meta: {rewrite_n:,}  ({100*rewrite_n/total:.1f}%)')
print(f'  -> review_snippet_intent : {review_n:,}  ({100*review_n/total:.1f}%)')
print(f'  -> monitor_ctr           : {monitor_n:,}  ({100*monitor_n/total:.1f}%)')
print(f'  No-action                : {total-actionable:,}  ({100*(total-actionable)/total:.1f}%)')
print('\nSection 2 complete.')


-- Intended Use Statement -------------------------------------------------------
This priority queue is a directional decision-support tool for editorial triage.
It ranks pages by observed CTR opportunity and model confidence, but does not
guarantee that a rewrite will cause improvement. All recommendations require
human review before action.

Validation scope  : 28 clients, Jan-Feb 2026 dev window, >=100 impression floor.
Not valid for     : navigational/branded queries, pages outside the observed
                    volume range, or sites in verticals absent from this release.
Language          : findings are observational and directional; no causal claims.
--------------------------------------------------------------------------------

Scope summary
  Total pages scored       : 60,088
  Actionable (any tier)    : 29,291  (48.7%)
  -> rewrite_title_and_meta: 2,883  (4.8%)
  -> review_snippet_intent : 10,038  (16.7%)
  -> monitor_ctr           : 16,370  (27.2%)
  No-action         

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### What a reviewer must check before acting

Every recommended page requires a human to verify the following **before** writing a new title or meta description:

| Check | Why |
|---|---|
| **Is the page still live and indexed?** | The queue was built on historical data — a page may have been deleted, redirected, or de-indexed since the feature window |
| **Has a rewrite already happened in this cycle?** | Avoid double-editing; check the CMS for any title or meta changes made after the feature window closes |
| **Does the current SERP show a rich result or AI Overview?** | If a SERP feature dominates the top of the page, a title rewrite is unlikely to recover organic clicks — triage this separately |
| **Is the primary search intent still informational/transactional?** | Intent drift (e.g. a query turned commercial) means the optimal title angle has changed — verify before writing |
| **Does the new title fit brand and editorial guidelines?** | Every rewrite must clear a style/brand review regardless of the model's recommendation |

### The no-go list — what must never be automated

| No-go | Reason |
|---|---|
| **Bulk-rewriting without per-page human review** | One misaligned rewrite can tank rankings for a high-traffic page; batch automation carries disproportionate risk |
| **Acting on pages with < 100 impressions** | These were excluded from the model's valid range; CTR is too noisy to produce a reliable signal |
| **Applying this queue to branded/navigational pages** | The model was not trained on these query types — output is unreliable and may actively mislead |
| **Treating the RF probability as a guaranteed outcome** | It is a probability derived from observational data, not a controlled experiment. Use it to prioritise, not to promise |
| **Rewriting `no_action` pages to chase higher scores** | Pages in this tier are at or above their benchmark — unprompted rewrites risk disrupting what is already working |

In [7]:
# -- 3. Sanity checks + reviewer reference card -------------------------------
action_queue = queue[queue['action_label'] != 'no_action'].copy()

assert (action_queue['action_label'] == 'no_action').sum() == 0
rw_rows = action_queue[action_queue['action_label'] == 'rewrite_title_and_meta']
assert (rw_rows['rf_prob'] >= 0.70).all()
assert rw_rows['position_tier'].isin(['top_3', 'striking']).all()
assert (rw_rows['opportunity_gap'] > 0.50).all()

print('-- Pre-action reviewer checklist ----------------------------------------')
checklist = [
    'Is the page still live and indexed?',
    'Has a rewrite already happened in this cycle?',
    'Does the current SERP show a rich result or AI Overview?',
    'Is the primary search intent still informational/transactional?',
    'Does the proposed title fit brand and editorial guidelines?',
]
for i, item in enumerate(checklist, 1):
    print(f'  {i}. [ ] {item}')
print()
print('-- No-go list -----------------------------------------------------------')
nogo = [
    'Bulk-rewriting without per-page human review',
    'Acting on pages with < 100 impressions (outside model valid range)',
    'Applying this queue to branded/navigational pages',
    'Treating RF probability as a guaranteed outcome',
    'Rewriting no_action pages to chase higher scores',
]
for item in nogo:
    print(f'  [X] {item}')
print()
print(f'Sanity checks passed: {len(action_queue):,} actionable pages, 0 no_action leakage')
print('\nSection 3 complete.')

-- Pre-action reviewer checklist ----------------------------------------
  1. [ ] Is the page still live and indexed?
  2. [ ] Has a rewrite already happened in this cycle?
  3. [ ] Does the current SERP show a rich result or AI Overview?
  4. [ ] Is the primary search intent still informational/transactional?
  5. [ ] Does the proposed title fit brand and editorial guidelines?

-- No-go list -----------------------------------------------------------
  [X] Bulk-rewriting without per-page human review
  [X] Acting on pages with < 100 impressions (outside model valid range)
  [X] Applying this queue to branded/navigational pages
  [X] Treating RF probability as a guaranteed outcome
  [X] Rewriting no_action pages to chase higher scores

Sanity checks passed: 29,291 actionable pages, 0 no_action leakage

Section 3 complete.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### What makes the queue go stale

Three conditions can silently invalidate the priority queue:

**1. CTR improvement rate drops in production**
The model was calibrated on a 59% base rate of pages improving CTR by >=10%. If pages acted on show a real-world improvement rate well below this — indicating the model's confidence is no longer calibrated — it's time to retrain.

**2. Tier benchmarks shift**
The `tier_expected_ctr` values (e.g. top_3 = 2.76%) are fixed snapshots from January 2026. SERP layout changes (more AI Overviews, more zero-click results) can compress organic CTR across all tiers, making the opportunity gap estimates systematically wrong.

**3. Feature distributions drift**
When new data months arrive, the distribution of `feat_ctr` and `avg_position` may shift. If those shifts are large enough to be statistically significant, the model is operating outside its training distribution and should be retrained on fresh data.

### Retrain triggers — any one is sufficient

| Trigger | Threshold | Rationale |
|---|---|---|
| **Production CTR improvement rate** | Drops below **45%** (was 59% at training) | > 14 pp drop signals meaningful model miscalibration |
| **Top-3 tier CTR benchmark shift** | Moves outside **2.35% – 3.18%** (+-15% relative of 2.76%) | Benchmark drift makes opportunity gap estimates unreliable |
| **Feature distribution drift** | KS-test p-value < **0.05** for `feat_ctr` or `avg_position` on a new data month | Statistical evidence the model is out-of-distribution |
| **Calendar trigger** | Every **90 days** regardless of the above | Prevents silent drift; aligns with quarterly content review cycles |

In [8]:
# -- 4. Monitoring reference card ---------------------------------------------
from scipy import stats

TRAIN_BASE_RATE    = df['ctr_improved'].mean()
TRAIN_TOP3_CTR     = df.loc[df['position_tier'] == 'top_3', 'tier_expected_ctr'].iloc[0]
TRIGGER_IMPROVEMENT_RATE = TRAIN_BASE_RATE - 0.14
TRIGGER_TOP3_LOW   = TRAIN_TOP3_CTR * 0.85
TRIGGER_TOP3_HIGH  = TRAIN_TOP3_CTR * 1.15

half = len(df) // 2
ks_feat_ctr = stats.ks_2samp(df['feat_ctr'].iloc[:half], df['feat_ctr'].iloc[half:])
ks_avg_pos  = stats.ks_2samp(df['avg_position'].iloc[:half], df['avg_position'].iloc[half:])

print('-- Monitoring Reference Card (computed from training data) --------------')
print(f'  Training base rate (CTR improved)  : {100*TRAIN_BASE_RATE:.1f}%')
print(f'  Retrain if production rate drops to: {100*TRIGGER_IMPROVEMENT_RATE:.1f}%  (>14 pp drop)')
print()
print(f'  Training top-3 expected CTR        : {TRAIN_TOP3_CTR:.4f}%')
print(f'  Retrain if benchmark exits range   : [{TRIGGER_TOP3_LOW:.4f}%, {TRIGGER_TOP3_HIGH:.4f}%]')
print()
print(f'  KS drift threshold (p-value)       : < 0.05')
print(f'  Training data self-check:')
print(f'    feat_ctr  KS p-value : {ks_feat_ctr.pvalue:.4f}  -> {"PASS" if ks_feat_ctr.pvalue > 0.05 else "DRIFT DETECTED"}')
print(f'    avg_pos   KS p-value : {ks_avg_pos.pvalue:.4f}  -> {"PASS" if ks_avg_pos.pvalue > 0.05 else "DRIFT DETECTED"}')
print()
print(f'  Calendar trigger                   : retrain every 90 days regardless')
print()
print('-- How to run this check on new data ------------------------------------')
print('  1. Load new month parquet into new_df')
print('  2. Run: stats.ks_2samp(df["feat_ctr"], new_df["feat_ctr"])')
print('  3. If p-value < 0.05, rebuild the cache and retrain')
print('  4. Compare new production improvement rate against the threshold above')
print('\nSection 4 complete.')

-- Monitoring Reference Card (computed from training data) --------------
  Training base rate (CTR improved)  : 59.0%
  Retrain if production rate drops to: 45.0%  (>14 pp drop)

  Training top-3 expected CTR        : 2.7645%
  Retrain if benchmark exits range   : [2.3498%, 3.1792%]

  KS drift threshold (p-value)       : < 0.05
  Training data self-check:
    feat_ctr  KS p-value : 0.3252  -> PASS
    avg_pos   KS p-value : 0.9469  -> PASS

  Calendar trigger                   : retrain every 90 days regardless

-- How to run this check on new data ------------------------------------
  1. Load new month parquet into new_df
  2. Run: stats.ks_2samp(df["feat_ctr"], new_df["feat_ctr"])
  3. If p-value < 0.05, rebuild the cache and retrain
  4. Compare new production improvement rate against the threshold above

Section 4 complete.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### What this section exports

Five artefacts written to `work/outputs/` that the deployed research paper will embed directly:

| File | Used in paper section |
|---|---|
| `action_queue_top100.csv` | Ranked Recommendations — the paper's action table |
| `action_distribution.png` | Results — page count per action label (bar chart) |
| `priority_score_by_tier.png` | Results — priority score distribution by position tier (box plot) |
| `rf_prob_distribution.png` | Methodology — RF probability score histogram |
| `playbook_summary.json` | Abstract + Results — inline numbers (total scored, rewrite count, base rate) |

In [9]:
# -- 5a. Export: action_queue_top100.csv --------------------------------------
EXPORT_COLS = [
    'queue_rank', 'content_hash_id', 'position_tier', 'avg_position',
    'total_impressions', 'feat_ctr', 'tier_expected_ctr', 'opportunity_gap',
    'rf_prob', 'priority_score', 'action_label', 'reason_text'
]
top100 = queue[EXPORT_COLS].head(100)
csv_path = OUTPUTS / 'action_queue_top100.csv'
top100.to_csv(csv_path, index=False)
print(f'Saved: {csv_path.name}  ({csv_path.stat().st_size:,} bytes)  [{len(top100)} rows]')

Saved: action_queue_top100.csv  (28,531 bytes)  [100 rows]


In [10]:
# -- 5b. Chart 1: action distribution (horizontal bar chart) ------------------
PALETTE = {
    'rewrite_title_and_meta'   : '#e05a5a',
    'review_snippet_and_intent': '#e89c4a',
    'monitor_ctr'              : '#5a9ee0',
    'no_action'                : '#9cacb4',
}
LABEL_DISPLAY = {
    'rewrite_title_and_meta'   : 'Rewrite title & meta',
    'review_snippet_and_intent': 'Review snippet & intent',
    'monitor_ctr'              : 'Monitor CTR',
    'no_action'                : 'No action',
}

dist_df = (
    df['action_label'].value_counts()
    .reindex(['rewrite_title_and_meta', 'review_snippet_and_intent',
              'monitor_ctr', 'no_action'])
    .reset_index()
)
dist_df.columns = ['action', 'count']

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bars = ax.barh(
    [LABEL_DISPLAY[a] for a in dist_df['action']],
    dist_df['count'],
    color=[PALETTE[a] for a in dist_df['action']],
    height=0.55,
    edgecolor='none',
)
for bar, n in zip(bars, dist_df['count']):
    ax.text(bar.get_width() + 150, bar.get_y() + bar.get_height()/2,
            f'{n:,}', va='center', ha='left', color='#cccccc', fontsize=10)

ax.set_xlabel('Number of pages', color='#cccccc', fontsize=10)
ax.set_title('Action Distribution across 60 k scored pages', color='white', fontsize=12, pad=12)
ax.tick_params(colors='#cccccc', labelsize=10)
ax.spines[['top','right','bottom','left']].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlim(0, dist_df['count'].max() * 1.18)
plt.tight_layout()

dist_path = OUTPUTS / 'action_distribution.png'
fig.savefig(dist_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print(f'Saved: {dist_path.name}  ({dist_path.stat().st_size:,} bytes)')

Saved: action_distribution.png  (38,918 bytes)


In [11]:
# -- 5c. Chart 2: priority score by position tier (box plot) ------------------
TIER_ORDER  = ['top_3', 'striking', 'page_1', 'page_3_5', 'deep']
TIER_LABELS = ['Top 3', 'Striking\ndistance', 'Page 1', 'Pages 3-5', 'Deep\n(pos > 20)']
TIER_COLORS = ['#e05a5a', '#e89c4a', '#5a9ee0', '#7ec8a4', '#9cacb4']

plot_data = [df.loc[df['position_tier'] == t, 'priority_score'].dropna().values
             for t in TIER_ORDER]

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bp = ax.boxplot(
    plot_data,
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(color='#aaaaaa'),
    capprops=dict(color='#aaaaaa'),
    flierprops=dict(marker='o', markersize=2, color='#555555', alpha=0.4),
    widths=0.5,
)
for patch, color in zip(bp['boxes'], TIER_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(TIER_LABELS, color='#cccccc', fontsize=10)
ax.set_ylabel('Priority score', color='#cccccc', fontsize=10)
ax.set_title('Priority Score Distribution by Position Tier', color='white', fontsize=12, pad=12)
ax.tick_params(axis='y', colors='#cccccc', labelsize=9)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#444444')
ax.yaxis.grid(True, color='#2a2a2a', linestyle='--', linewidth=0.6)
ax.set_axisbelow(True)
plt.tight_layout()

tier_path = OUTPUTS / 'priority_score_by_tier.png'
fig.savefig(tier_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print(f'Saved: {tier_path.name}  ({tier_path.stat().st_size:,} bytes)')

Saved: priority_score_by_tier.png  (38,575 bytes)


In [12]:
# -- 5d. Chart 3: RF probability distribution (histogram) ---------------------
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

ax.hist(df['rf_prob'], bins=40, color='#5a9ee0', alpha=0.85, edgecolor='none')
ax.axvline(0.70, color='#e05a5a', linewidth=1.5, linestyle='--', label='Rewrite threshold (0.70)')
ax.axvline(0.50, color='#e89c4a', linewidth=1.5, linestyle='--', label='Monitor threshold (0.50)')

ax.set_xlabel('RF probability score', color='#cccccc', fontsize=10)
ax.set_ylabel('Number of pages', color='#cccccc', fontsize=10)
ax.set_title('Random Forest Probability Score Distribution', color='white', fontsize=12, pad=12)
ax.tick_params(colors='#cccccc', labelsize=9)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#444444')
ax.yaxis.grid(True, color='#2a2a2a', linestyle='--', linewidth=0.6)
ax.set_axisbelow(True)
legend = ax.legend(fontsize=9, facecolor='#1a1a2e', labelcolor='#cccccc', framealpha=0.8)
plt.tight_layout()

hist_path = OUTPUTS / 'rf_prob_distribution.png'
fig.savefig(hist_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print(f'Saved: {hist_path.name}  ({hist_path.stat().st_size:,} bytes)')

Saved: rf_prob_distribution.png  (42,312 bytes)


In [13]:
# -- 5e. Export: playbook_summary.json ----------------------------------------
import json as _json

summary = {
    'total_pages_scored'        : int(len(df)),
    'clients'                   : int(df['client_hash_id'].nunique()),
    'observation_window'        : 'Jan-Feb 2026 (dev)',
    'impression_floor'          : 100,
    'action_counts': {
        'rewrite_title_and_meta'   : int((df['action_label'] == 'rewrite_title_and_meta').sum()),
        'review_snippet_and_intent': int((df['action_label'] == 'review_snippet_and_intent').sum()),
        'monitor_ctr'              : int((df['action_label'] == 'monitor_ctr').sum()),
        'no_action'                : int((df['action_label'] == 'no_action').sum()),
    },
    'model': {
        'type'                     : 'RandomForestClassifier',
        'n_estimators'             : 100,
        'max_depth'                : 6,
        'min_samples_leaf'         : 20,
        'features'                 : FEATURES,
        'label'                    : 'ctr_improved (>=10% CTR gain Jan->Feb)',
        'train_base_rate_pct'      : round(float(df['ctr_improved'].mean()) * 100, 1),
    },
    'priority_score_formula'    : 'rf_prob x opportunity_gap x log1p(total_impressions)',
    'monitoring': {
        'retrain_if_improvement_rate_below_pct': 45.0,
        'retrain_if_top3_ctr_outside_range'    : [round(float(df.loc[df['position_tier']=='top_3','tier_expected_ctr'].iloc[0])*0.85,4),
                                                   round(float(df.loc[df['position_tier']=='top_3','tier_expected_ctr'].iloc[0])*1.15,4)],
        'retrain_if_ks_pvalue_below'           : 0.05,
        'calendar_retrain_every_days'          : 90,
    },
    'exports': [
        'action_queue_top100.csv',
        'action_distribution.png',
        'priority_score_by_tier.png',
        'rf_prob_distribution.png',
        'playbook_summary.json',
    ]
}

json_path = OUTPUTS / 'playbook_summary.json'
json_path.write_text(_json.dumps(summary, indent=2))
print(f'Saved: {json_path.name}  ({json_path.stat().st_size:,} bytes)')
print()
print('-- All exports complete -------------------------------------------------')
for f in ['action_queue_top100.csv', 'action_distribution.png',
          'priority_score_by_tier.png', 'rf_prob_distribution.png',
          'playbook_summary.json']:
    p = OUTPUTS / f
    status = f'{p.stat().st_size:,} bytes' if p.exists() else 'MISSING'
    print(f'  {f:<35s}: {status}')
print('\nSection 5 complete.  Notebook done.')

Saved: playbook_summary.json  (1,133 bytes)

-- All exports complete -------------------------------------------------
  action_queue_top100.csv            : 28,531 bytes
  action_distribution.png            : 38,918 bytes
  priority_score_by_tier.png         : 38,575 bytes
  rf_prob_distribution.png           : 42,312 bytes
  playbook_summary.json              : 1,133 bytes

Section 5 complete.  Notebook done.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.